# Interactive Slip Distribution Mapping

This notebook provides an interactive interface to generate and inspect slip distribution maps produced by pyANTI-FASc.

The main numerical workflow should be executed beforehand using the command-line interface.  
This notebook is intended for post-processing and visualization only.

## Notes

This notebook is designed for visualization and post-processing.

For best performance:

- Run the full pyANTI-FASc pipeline from the command line
- Use this notebook only after the output files have been generated
- Avoid re-running map generation unless the input slip files have changed

The first cells can be used to visualize the maps through the geojson files already computed by the code
The other ones might be used to generate interactive html files and visualize them


## 1. Imports and configuration

In this section, we import the required Python modules and load the pyANTI-FASc configuration file.

The plotting functions are imported from the local Python script:

```text
plot_slip_dsitribution.py
```

The configuration file is read from:

```text
../config_files/Parameters/input.json
```

The output will be searched among the results in the folder:

```text
../output
```



In [5]:
import json
from pathlib import Path
import shutil
import ipywidgets as widgets
from IPython.display import display, clear_output, IFrame
import folium
from branca.colormap import LinearColormap

from utils.plot_slip_distribution import ascii_to_geojson, plot_slip_map, progress, plot_geojson_property, generate_slip_maps

with open("../config_files/Parameters/input.json") as fid:
    Param = json.load(fid)

base_folder = Path("../output")

## 2. Select the output directory

The pyANTI-FASc output directory contains a nested folder structure describing the selected run.

The following widgets allow the user to select:

1. Event
2. Rigidity distribution
3. Magnitude
4. Scaling law

Within the final selected folder, select the slip distribution you want to plot

In [6]:
levels = [
    "event",
    "rigidity distribution",
    "magnitude",
    "scaling law",
]

out = widgets.Output()

folder_dropdowns = []

geojson_dropdown = widgets.Dropdown(
    options=[],
    description="GeoJSON:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="650px")
)

html_dropdown = widgets.Dropdown(
    options=[],
    description="HTML map:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="650px")
)

html_dropdown_box = widgets.VBox([])

property_dropdown = widgets.Dropdown(
    options=["slip", "rake"],
    value="slip",
    description="Feature:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="350px")
)

tile_options = {
    "CartoDB Positron": {
        "tiles": "CartoDB positron",
        "attr": None
    },
    "CartoDB Dark Matter": {
        "tiles": "CartoDB dark_matter",
        "attr": None
    },
    "CartoDB Voyager": {
        "tiles": "CartoDB Voyager",
        "attr": None
    },
    "CartoDB Positron No Labels": {
        "tiles": "CartoDB positron_nolabels",
        "attr": None
    },
    "OpenTopoMap": {
        "tiles": "https://{s}.tile.opentopomap.org/{z}/{x}/{y}.png",
        "attr": "Map data: © OpenStreetMap contributors, SRTM | Map style: © OpenTopoMap (CC-BY-SA)"
    },
    "Esri Satellite + Labels": {
        "type": "esri_hybrid"
    }
}

tile_dropdown = widgets.Dropdown(
    options=list(tile_options.keys()),
    value="CartoDB Positron",
    description="Basemap:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="500px")
)


plot_button = widgets.Button(
    description="Plot GeoJSON",
    button_style="success"
)

def list_dirs(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted([p.name for p in folder.iterdir() if p.is_dir()])

def selected_folder():
    folder = base_folder
    for dd in folder_dropdowns:
        if dd.value is not None:
            folder = folder / dd.value
    return folder

def build_folder_until(index):
    folder = base_folder
    for i in range(index):
        value = folder_dropdowns[i].value
        if value is not None:
            folder = folder / value
    return folder

def update_geojson_dropdown():
    folder = selected_folder()
    geojson_files = sorted(folder.glob("Slip4H*.json"))

    geojson_dropdown.options = [f.name for f in geojson_files]

    if geojson_files:
        geojson_dropdown.value = geojson_files[0].name
    else:
        geojson_dropdown.value = None

def update_folder_dropdowns(change=None):
    for i, dd in enumerate(folder_dropdowns):
        parent = build_folder_until(i)
        choices = list_dirs(parent)

        old_value = dd.value
        dd.options = choices

        if old_value in choices:
            dd.value = old_value
        elif choices:
            dd.value = choices[0]
        else:
            dd.value = None

    update_geojson_dropdown()

def update_html_dropdown():
    folder = selected_folder()
    html_files = sorted(folder.glob("Slip4H*.html"))
    html_dropdown.options = [f.name for f in html_files]

    if html_files:
        html_dropdown.value = html_files[0].name
    else:
        html_dropdown.value = None


def show_html_map(change=None):
    with out:
        clear_output()

        folder = selected_folder()

        if not html_dropdown.value:
            print(f"No HTML maps found in: {folder}")
            return

        src_file = folder / html_dropdown.value
        preview_file = Path("preview_map.html")

        shutil.copy(src_file, preview_file)

        print(f"Showing: {src_file}")
        display(IFrame("preview_map.html", width=950, height=650))

for level in levels:
    dd = widgets.Dropdown(
        description=level + ":",
        style={"description_width": "160px"},
        layout=widgets.Layout(width="650px")
    )
    dd.observe(update_folder_dropdowns, names="value")
    folder_dropdowns.append(dd)



def on_plot_clicked(button):
    with out:
        clear_output()

        folder = selected_folder()

        if not geojson_dropdown.value:
            print(f"No GeoJSON files found in: {folder}")
            return

        geojson_file = folder / geojson_dropdown.value
        property_name = property_dropdown.value
        tile_config = tile_options[tile_dropdown.value]

        print(f"Selected folder: {folder}")
        print(f"GeoJSON: {geojson_file.name}")
        print(f"Feature: {property_name}")
        print(f"Basemap: {tile_dropdown.value}")

        m = plot_geojson_property(geojson_file, property_name,tile_config=tile_config)

        if m is not None:
            display(m)

html_button = widgets.Button(
    description="Generate HTML maps",
    button_style="warning",
    layout=widgets.Layout(width="180px")
)

def on_html_clicked(button):
    with out:
        clear_output()

        folder = selected_folder()
        print(f"Selected folder: {folder}")
        print("Generating HTML maps...")

        generate_slip_maps(folder, Param)

        print("Done.")
        update_geojson_dropdown()
        update_html_dropdown()

        html_dropdown_box.children = [html_dropdown]

        if html_dropdown.value:
            show_html_map()

html_button.on_click(on_html_clicked)

plot_button.on_click(on_plot_clicked)

update_folder_dropdowns()

html_dropdown.observe(show_html_map, names="value")

display(
    *folder_dropdowns,
    geojson_dropdown,
    property_dropdown,
    tile_dropdown,
    widgets.HBox([plot_button, html_button]),
    html_dropdown_box,
    out
)

Dropdown(description='event:', layout=Layout(width='650px'), options=('.ipynb_checkpoints', 'Tohoku_test_delet…

Dropdown(description='rigidity distribution:', layout=Layout(width='650px'), options=(), style=DescriptionStyl…

Dropdown(description='magnitude:', layout=Layout(width='650px'), options=(), style=DescriptionStyle(descriptio…

Dropdown(description='scaling law:', layout=Layout(width='650px'), options=(), style=DescriptionStyle(descript…

Dropdown(description='GeoJSON:', layout=Layout(width='650px'), options=(), style=DescriptionStyle(description_…

Dropdown(description='Feature:', layout=Layout(width='350px'), options=('slip', 'rake'), style=DescriptionStyl…

Dropdown(description='Basemap:', layout=Layout(width='500px'), options=('CartoDB Positron', 'CartoDB Dark Matt…

VBox()

Output()

## 3. ⬆️🙄 Generate slip distribution maps 🙄⬆️

After selecting the desired output folder, click one of the button below to generate the slip distribution maps.

For each selection, the notebook will:
1. Plot the selected GeoJSON (visualizing either slip or rake) 
2. Generate an interactive Folium HTML map and save the resulting `.html` file in the same output folder

**If you have generated HTML maps, a new dropdown menu will appear. The user can select from there the slip distributions to visualize**



## 4. Download and clean output files

Use the widget below to compress one selected pyANTI-FASc output folder into a `.tar.gz` archive.

After creating the archive, the notebook prints the exact archive path. To download it, use the JupyterLab file browser: go to that path, right-click the `.tar.gz` file, and select **Download**.

Because Jupyter cannot reliably detect when the browser has completed a file download, cleanup requires an explicit confirmation:

1. Create the archive.
2. Download it manually from the JupyterLab file browser.
3. Check that the download has completed successfully on your local machine.
4. Type `DELETE` in the confirmation box and click the red cleanup button.

The cleanup action removes both the selected output folder and the generated `.tar.gz` archive from the platform.


In [11]:
from pathlib import Path
import shutil
import tarfile
import time

import ipywidgets as widgets
from IPython.display import display, clear_output

# This notebook is expected to run from the bin/ directory.
# pyANTI-FASc outputs are normally stored in ../output.
OUTPUT_ROOT = Path("../output").resolve()
ARCHIVE_DIR = Path("download_archives").resolve()
ARCHIVE_DIR.mkdir(exist_ok=True)

state = {
    "last_archive": None,
    "last_selected_folder": None,
}

def _is_inside(path, root):
    """Return True only if path is inside root."""
    path = Path(path).resolve()
    root = Path(root).resolve()
    try:
        path.relative_to(root)
        return True
    except ValueError:
        return False


def _list_output_folders():
    """List direct output folders available under OUTPUT_ROOT."""
    if not OUTPUT_ROOT.exists():
        return []

    folders = sorted(
        [p for p in OUTPUT_ROOT.iterdir() if p.is_dir()],
        key=lambda p: p.name.lower()
    )

    return [(p.name, str(p.resolve())) for p in folders]


output_dropdown = widgets.Dropdown(
    options=_list_output_folders(),
    description="Output folder:",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="700px")
)

refresh_button = widgets.Button(
    description="Refresh folders",
    button_style="",
    tooltip="Refresh the list of output folders"
)

archive_button = widgets.Button(
    description="Create output archive",
    button_style="primary",
    tooltip="Create a .tar.gz archive for the selected output folder",
    layout=widgets.Layout(width="200px")
)

cleanup_button = widgets.Button(
    description="Clean output from platform",
    button_style="danger",
    tooltip="Open the cleanup confirmation panel",
    layout=widgets.Layout(width="200px",height="40px")
)

confirm_delete_button = widgets.Button(
    description="Yes, delete output and archive",
    button_style="danger",
    tooltip="Permanently remove the selected output folder and generated archive from the platform",
    layout=widgets.Layout(width="200px",height="42px")
    
)

cancel_delete_button = widgets.Button(
    description="Cancel",
    button_style="",
    tooltip="Cancel cleanup"
)

status = widgets.Output()
confirm_box = widgets.Output()


def refresh_folders(_=None):
    current = output_dropdown.value
    output_dropdown.options = _list_output_folders()

    values = [v for _, v in output_dropdown.options]
    if current in values:
        output_dropdown.value = current

    with status:
        clear_output()
        print(f"Output root: {OUTPUT_ROOT}")
        if len(output_dropdown.options) == 0:
            print("No output folders found.")
        else:
            print(f"Found {len(output_dropdown.options)} output folder(s).")


def create_archive(_):
    with confirm_box:
        clear_output()

    with status:
        clear_output()

        if not output_dropdown.value:
            print("No output folder selected.")
            return

        selected = Path(output_dropdown.value).resolve()

        if not _is_inside(selected, OUTPUT_ROOT):
            raise RuntimeError(f"Refusing to archive path outside OUTPUT_ROOT: {selected}")

        if not selected.exists() or not selected.is_dir():
            print(f"Selected folder does not exist: {selected}")
            return

        timestamp = time.strftime("%Y%m%d_%H%M%S")
        archive_path = ARCHIVE_DIR / f"{selected.name}_{timestamp}.tar.gz"

        print("Creating archive for:")
        print(selected)
        print()
        print("Archive:")
        print(archive_path)

        with tarfile.open(archive_path, "w:gz") as tar:
            tar.add(selected, arcname=selected.name)

        state["last_archive"] = archive_path
        state["last_selected_folder"] = selected

        print()
        print("Archive created.")
        print()
        print("Archive path:")
        print(archive_path)
        print()
        print("To download it from JupyterLab:")
        print("1. Open the file browser on the left.")
        print("2. Go to the folder shown above.")
        print("3. Right-click the .tar.gz file.")
        print("4. Select Download.")
        print()
        print("After checking that the download has completed successfully,")
        print("you may use the red cleanup button to remove the platform copy.")


def request_cleanup_confirmation(_):
    with status:
        clear_output()

        archive_path = state.get("last_archive")
        selected = state.get("last_selected_folder")

        if archive_path is None or selected is None:
            print("No archive has been created in this session yet.")
            return

        archive_path = Path(archive_path).resolve()
        selected = Path(selected).resolve()

        if not _is_inside(selected, OUTPUT_ROOT):
            raise RuntimeError(f"Refusing to delete path outside OUTPUT_ROOT: {selected}")

    with confirm_box:
        clear_output()
        display(widgets.HTML(
            "<div style='border: 2px solid #b00020; padding: 12px; border-radius: 6px;'>"
            "<b>Confirm cleanup</b><br><br>"
            "Before continuing, check that the <code>.tar.gz</code> archive has been downloaded "
            "successfully to your local machine.<br><br>"
            "If you confirm, the notebook will remove <b>both</b> of the following from the platform:"
            "</div>"
        ))
        print()
        print("Output folder:")
        print(selected)
        print()
        print("Archive:")
        print(archive_path)
        print()
        display(widgets.HBox([confirm_delete_button, cancel_delete_button]))


def perform_cleanup(_):
    with confirm_box:
        clear_output()

    with status:
        clear_output()

        archive_path = state.get("last_archive")
        selected = state.get("last_selected_folder")

        if archive_path is None or selected is None:
            print("No archive has been created in this session yet.")
            return

        archive_path = Path(archive_path).resolve()
        selected = Path(selected).resolve()

        if not _is_inside(selected, OUTPUT_ROOT):
            raise RuntimeError(f"Refusing to delete path outside OUTPUT_ROOT: {selected}")

        print("Cleanup confirmed.")
        print("Deleting output folder and archive from the platform...")
        print()

        if selected.exists():
            print("Deleting output folder:")
            print(selected)
            shutil.rmtree(selected)
        else:
            print("Output folder already removed:")
            print(selected)

        if archive_path.exists():
            print()
            print("Deleting archive:")
            print(archive_path)
            archive_path.unlink()
        else:
            print()
            print("Archive already removed:")
            print(archive_path)

        state["last_archive"] = None
        state["last_selected_folder"] = None

        print()
        print("Cleanup completed.")
        refresh_folders()


def cancel_cleanup(_):
    with confirm_box:
        clear_output()
    with status:
        clear_output()
        print("Cleanup cancelled. No files were removed.")


refresh_button.on_click(refresh_folders)
archive_button.on_click(create_archive)
cleanup_button.on_click(request_cleanup_confirmation)
confirm_delete_button.on_click(perform_cleanup)
cancel_delete_button.on_click(cancel_cleanup)

display(widgets.VBox([
    widgets.HTML("<b>Compress and download pyANTI-FASc outputs</b>"),
    widgets.HTML(
        "Select an output folder, create a <code>.tar.gz</code> archive, "
        "download it manually from the JupyterLab file browser, and then delete "
        "the platform copy only after the second confirmation step."
    ),
    output_dropdown,
    widgets.HBox([refresh_button, archive_button]),
    cleanup_button,
    confirm_box,
    status
]))

refresh_folders()
